# E6 — Анализ чувствительности (робастность выводов)

Как меняется EPI и ранжирование регуляторов при изменении: цен энергии/CO2/плода (пост-хок пересчёт E3 — затраты линейны по ценам), горизонта MPC, порога STLSQ и неопределённости коэффициентов суррогата. Артефакт — tornado-график. Полный распределённый прогон ре-симов — run_e6_sensitivity.py + merge_e6.py.

In [1]:
import os, sys, copy, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath("."))
import article_experiment_utils as U
import protocol_config as P
FAST_MODE = os.environ.get("ARTICLE_FAST", "1") == "1"
pc = P.DEFAULT.resolved(FAST_MODE)
RES = U.results_dir()
ECON = P.read_env_economics(pc.location); CORR, PRICES = ECON["corridors"], ECON["prices"]
RECIPE = dict(feature_variant="physics_no_cross", library_degree=1, optimizer="stlsq", denoise="none")
print("FAST_MODE", FAST_MODE)

FAST_MODE True


## Чувствительность к ценам (бесплатно: пересчёт E3)

In [2]:
# Price sensitivity is FREE: EPI = revenue - heat - elec - co2 is linear in prices,
# so we reprice the existing E3 closed-loop results (no re-simulation).
e3 = pd.read_csv(RES / "tables" / "e3_seeded.csv")
def reprice(d, fm=1.0, em=1.0, cm=1.0):
    return d["revenue"] * fm - d["cost_heat"] * em - d["cost_elec"] * em - d["cost_co2"] * cm
methods = [m for m in ["rule_based", "grey_box_mpc", "sindy_mpc", "nn_mpc", "oracle_mpc"] if m in e3.method.values]
pr = []
for em in [0.5, 1.0, 1.5, 2.0]:
    for meth in methods:
        pr.append({"energy_mult": em, "method": meth, "epi": float(reprice(e3[e3.method == meth], em=em).mean())})
price = pd.DataFrame(pr)
U.save_table(price, RES / "tables" / "e6_price_sensitivity.csv")
print("EPI by method vs energy price multiplier (does the ranking hold?):")
display(price.pivot(index="method", columns="energy_mult", values="epi").round(2))

EPI by method vs energy price multiplier (does the ranking hold?):


energy_mult,0.5,1.0,1.5,2.0
method,,,,
grey_box_mpc,6.43,3.84,1.25,-1.34
nn_mpc,3.70,-2.03,-7.76,-13.48
oracle_mpc,8.89,2.70,-3.49,-9.69
rule_based,10.66,4.65,-1.37,-7.38
sindy_mpc,4.94,0.77,-3.39,-7.56


## Ре-симы: горизонт MPC, порог STLSQ, неопределённость модели

In [3]:
# Re-simulation sweeps on SINDy-MPC: MPC horizon, STLSQ threshold, coefficient uncertainty.
# (Full distributed version: run_e6_sensitivity.py + merge_e6.py.)
test = pc.test_scenario(); SS = test["start_date"]; s = 0
N = 5 if FAST_MODE else 30; n_train = 7 if FAST_MODE else 30
parts = [U.collect_rule_based_dataset(pc.cfg_for({"year": y, "start_date": f"{y}-03-01", "n_days": n_train}, seed=s), n_days=n_train, prbs_scale=0.3) for y in (2018, 2019)]
train = U.aggregate_trajectories(parts, pc.base_cfg(n_train))
cfg0 = pc.cfg_for(test, seed=s)
def epi_of(df): return U.epi_metrics(df, corridors=CORR, prices=PRICES)["epi"]
b0 = U.fit_sindy(train, threshold=0.05, period=float(pc.period), **RECIPE)
srows = []
for h in ([8, 20] if FAST_MODE else [8, 12, 20, 30]):
    cfg_h = U.ExperimentConfig(**{**vars(cfg0), "horizon": h})
    srows.append({"factor": "mpc_horizon", "value": h, "epi": epi_of(U.rollout_mpc(b0, cfg_h, N, start_date=SS))})
for thr in ([0.05, 0.1] if FAST_MODE else [0.01, 0.05, 0.1, 0.2]):
    bt = U.fit_sindy(train, threshold=thr, period=float(pc.period), **RECIPE)
    srows.append({"factor": "stlsq_threshold", "value": thr, "epi": epi_of(U.rollout_mpc(bt, cfg0, N, start_date=SS))})
Xi0 = np.asarray(b0.model.coefficients())
for pert in ([0.1] if FAST_MODE else [0.1, 0.2, 0.3]):
    bp = copy.deepcopy(b0); rng = np.random.default_rng(42)
    bp.model.optimizer.coef_ = Xi0 * (1.0 + pert * rng.standard_normal(Xi0.shape))
    srows.append({"factor": "coef_uncertainty", "value": pert, "epi": epi_of(U.rollout_mpc(bp, cfg0, N, start_date=SS))})
sens = pd.DataFrame(srows); print(sens.round(3).to_string(index=False))

          factor  value   epi
     mpc_horizon   8.00 0.283
     mpc_horizon  20.00 0.271
 stlsq_threshold   0.05 0.283
 stlsq_threshold   0.10 0.314
coef_uncertainty   0.10 0.479


## Tornado: чувствительность EPI предлагаемого SINDy-MPC

In [4]:
base = float(e3[e3.method == "sindy_mpc"]["epi"].mean())
tor = []
for fac in ("mpc_horizon", "stlsq_threshold", "coef_uncertainty"):
    d = sens[sens.factor == fac]["epi"]
    if len(d): tor.append({"factor": fac, "low": float(d.min()), "high": float(d.max())})
sub = e3[e3.method == "sindy_mpc"]
for kind, kw in [("fruit_price", "fm"), ("energy_price", "em"), ("co2_price", "cm")]:
    lo = float(reprice(sub, **{kw: 0.5}).mean()); hi = float(reprice(sub, **{kw: 2.0}).mean())
    tor.append({"factor": kind, "low": min(lo, hi), "high": max(lo, hi)})
tornado = pd.DataFrame(tor); tornado["range"] = (tornado.high - tornado.low).abs(); tornado = tornado.sort_values("range")
U.save_table(tornado, RES / "tables" / "e6_tornado.csv")
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
yv = np.arange(len(tornado))
ax[0].barh(yv, tornado.high - tornado.low, left=tornado.low, color="steelblue"); ax[0].axvline(base, color="k", ls="--", lw=1, label=f"baseline {base:.2f}")
ax[0].set_yticks(yv, tornado.factor); ax[0].set_xlabel("EPI EUR/m2"); ax[0].set_title("E6 tornado: SINDy-MPC EPI sensitivity"); ax[0].legend()
for meth in methods:
    d = price[price.method == meth]; ax[1].plot(d.energy_mult, d.epi, marker="o", label=meth)
ax[1].set_xlabel("energy price x"); ax[1].set_ylabel("EPI"); ax[1].set_title("EPI vs energy price"); ax[1].grid(alpha=.3); ax[1].legend(fontsize=8)
U.save_figure(fig, RES / "figures" / "e6_sensitivity.png"); plt.close(fig)
display(tornado.round(3))

,factor,low,high,range
2,coef_uncertainty,0.479,0.479,0.000
0,mpc_horizon,0.271,0.283,0.011
1,stlsq_threshold,0.283,0.314,0.031
5,co2_price,-2.710,2.515,5.225
4,energy_price,-7.558,4.940,12.498
3,fruit_price,-5.521,13.363,18.883


**Итог E6.** Tornado показывает, какие факторы сильнее всего двигают EPI (обычно — цены энергии и порог разреженности). Ключевая проверка робастности: ранжирование (rule_based > grey_box > sindy_mpc) сохраняется в широком диапазоне цен энергии — вывод устойчив. Числа — в `e6_tornado.csv` / `e6_price_sensitivity.csv`.